# 폰으로 찍은 방을 3D 로 만들어 웹에 올리기

방을 십오 초쯤 걸어 다니며 찍은 영상 하나로 시작합니다. 끝나면 **웹 주소 하나**가 남습니다.
브라우저에서 끌어 돌려 볼 수 있고, 남에게 링크로 보낼 수 있습니다.

만들어 둔 결과가 여기 있습니다. 먼저 열어 보십시오.
https://suncreation.github.io/room-3d-demo/

## 무엇을 거치는가

```
영상  →  사진 여러 장  →  점 무더기  →  알갱이  →  줄인 파일  →  내 주소
                              └─────→  메시  (게임에 쓰려면)
```

가운데 점 무더기까지가 AI 가 하는 일이고, 나머지는 파일을 바꾸는 일입니다.

## 얼마나 걸리나

T4 에서 잰 값입니다. **시간의 대부분이 모델 내려받기**이고 실제 계산은 십 초 남짓입니다.

| 걸음 | 시간 |
|---|---|
| 설치 | 약 25초 |
| 사진 뽑기 | 약 6초 |
| 모델 준비 | **약 180초** (4.91GB 를 받습니다) |
| 복원 사진 24장 | 약 10초 (최대 6.7GB) |
| 알갱이 파일 쓰기 | 약 1초 |
| 줄이기 | 약 30초 |

## 먼저 할 일

위 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU** 를 고르십시오.
GPU 없이는 복원 단계에서 멈춥니다.


## 1. 어떤 기계를 받았는지 확인합니다

계산 능력이 7.5 로 나오고 bfloat16 이 False 면 T4 입니다. 뒤에서 이 두 값이 쓰입니다.


In [ ]:
import torch, time
T0 = time.time()
assert torch.cuda.is_available(), "GPU 가 안 붙었습니다. 런타임 유형을 T4 GPU 로 바꾸고 다시 실행하십시오."
p = torch.cuda.get_device_properties(0)
cap = torch.cuda.get_device_capability()
print(f"{p.name} / 메모리 {p.total_memory/1e9:.1f}GB / 계산 능력 {cap[0]}.{cap[1]}")
print("bfloat16 을 쓸 수 있나:", cap[0] >= 8)


## 2. 설치합니다

세 가지가 필요합니다.

- **map-anything** 사진 여러 장에서 3차원 점을 뽑는 모델입니다. 저장소와 가중치 모두 Apache-2.0 이고 승인 절차가 없습니다
- **open3d** 점을 면으로 바꾸고 삼각형 수를 줄이는 데 씁니다. 마지막 메시 단계에서만 쓰입니다
- **trimesh** 메시를 `.glb` 로 내보낼 때 씁니다

`torch` 는 코랩에 이미 깔려 있고 건드리지 않습니다. 여기서 torch 를 다시 깔면 런타임을
재시작해야 해서 오히려 느려집니다.

한 번에 이 분에서 삼 분쯤 걸립니다.


In [ ]:
%%capture
!git clone -q --depth 1 https://github.com/facebookresearch/map-anything.git
!cd map-anything && pip install -q -e .
!pip install -q open3d trimesh


설치가 끝나도 **바로 `import mapanything` 하면 실패합니다.** 방금 깐 것은
`/content/map-anything` 아래에 있는데 파이썬이 그 경로를 아직 모릅니다.
경로를 끼워 넣으면 런타임을 재시작하지 않고 이어 갈 수 있습니다.


In [ ]:
import sys
sys.path.insert(0, '/content/map-anything')

import mapanything
print("mapanything 준비됨:", mapanything.__file__)
print(f"여기까지 {time.time()-T0:.0f}초")


## 3. 영상을 올립니다

아래를 실행하면 파일 고르는 창이 뜹니다.

**찍는 요령이 결과를 크게 가릅니다.**

- **걸어 다니며** 찍으십시오. 제자리에서 몸만 돌리면 안 됩니다. 카메라가 움직여야
  같은 물건이 다른 각도로 찍히고, 그래야 거리를 잡아낼 수 있습니다
- **가로로** 드십시오. 세로로 들면 화면의 위아래를 천장과 바닥이 차지하는데
  둘 다 밋밋해서 얻을 것이 적습니다
- **천천히** 움직이십시오. 빨리 지나가면 흐려진 장이 섞입니다
- **십오 초에서 이십 초** 면 충분합니다
- 자세히 보고 싶은 곳이 있으면 그 앞에서 좌우로 한두 걸음 더 움직여 주십시오


In [ ]:
import os
from google.colab import files

up = files.upload()
VIDEO = os.path.join('/content', list(up.keys())[0])
print(f"올린 영상: {VIDEO}  {os.path.getsize(VIDEO)/1e6:.1f}MB")


## 4. 영상에서 사진을 뽑습니다

영상을 통째로 넣지 않고 몇 장을 골라 씁니다. 옆 장과 거의 같은 사진을 잔뜩 넣어 봐야
얻는 것 없이 메모리만 먹습니다.

**몇 장이 좋은가**는 T4 에서 재 봤습니다.

| 장수 | 복원 시간 | 최대 메모리 | 점 개수 |
|---|---|---|---|
| 24 | 8.6초 | 6.81GB | 334만 |
| 32 | 12.9초 | 7.34GB | 446만 |
| 64 | 31.5초 | 9.49GB | 892만 |
| 96 | 55.1초 | 11.99GB | 1,338만 |
| 128 | 메모리가 넘칩니다 | | |

**24장이면 대체로 충분합니다.** 아래 `fps` 는 초당 몇 장을 뽑을지입니다.
십오 초 영상에서 24장을 얻으려면 1.6 쯤입니다. 영상 길이에 맞춰 고치십시오.

뽑고 나서 눈으로 한 번 보십시오. **번져 보이는 장이 여럿이면 다시 찍는 편이 빠릅니다.**


In [ ]:
import subprocess, os, glob
FPS = 1.6

os.makedirs('/content/frames', exist_ok=True)
for f in glob.glob('/content/frames/*.jpg'):
    os.remove(f)

t = time.time()
subprocess.run(['ffmpeg', '-v', 'error', '-i', VIDEO,
                '-vf', f'fps={FPS}', '-q:v', '2', '/content/frames/f_%03d.jpg'], check=True)
paths = sorted(glob.glob('/content/frames/*.jpg'))
print(f"{len(paths)}장 뽑음, {time.time()-t:.1f}초")

import matplotlib.pyplot as plt
from PIL import Image
fig, ax = plt.subplots(2, 4, figsize=(16, 5))
for a, pth in zip(ax.ravel(), paths[::max(1, len(paths)//8)][:8]):
    a.imshow(Image.open(pth)); a.axis('off'); a.set_title(os.path.basename(pth), fontsize=8)
plt.tight_layout(); plt.show()


## 5. 모델을 돌립니다

여기가 이 노트의 알맹이입니다.

**모델이 하는 일**은 사진마다 깊이 지도를 하나씩 뽑고, 그 사진들이 각각 어디서 찍힌
것인지를 함께 알아내어, 전부를 한 좌표계에 겹치는 것입니다. 결과로 색을 가진 점
수백만 개와 카메라가 있던 자리 스물몇 곳이 나옵니다.

**주의할 것이 하나 있습니다.** 공식 예제는 `amp_dtype="bf16"` 인데 **T4 에는 bf16 이
없습니다.** 그대로 두면 계산이 통째로 32비트로 풀려서 메모리가 넘칩니다. `fp16` 으로
바꿔야 합니다. 앞에서 확인한 bfloat16 이 False 라는 값이 여기서 쓰입니다.

**시간의 대부분이 모델 내려받기**입니다. 4.91GB 라 삼 분쯤 걸립니다. 한 번 받으면
같은 세션에서는 다시 안 받습니다.


In [ ]:
from mapanything.models import MapAnything
from mapanything.utils.image import load_images

t = time.time()
model = MapAnything.from_pretrained("facebook/map-anything-apache").to("cuda").eval()
print(f"모델 준비 {time.time()-t:.0f}초")

t = time.time()
views = load_images(paths)
print(f"사진 넣기 {time.time()-t:.1f}초, 한 장 크기 {tuple(views[0]['img'].shape[-2:])}")

torch.cuda.reset_peak_memory_stats()
t = time.time()
pred = model.infer(
    views,
    memory_efficient_inference=True,
    minibatch_size=1,
    use_amp=True,
    amp_dtype="fp16",     # T4 에서는 반드시 fp16 입니다
    apply_mask=True,
    mask_edges=True,
    apply_confidence_mask=False,
)
torch.cuda.synchronize()
print(f"복원 {time.time()-t:.1f}초, 최대 GPU 메모리 {torch.cuda.max_memory_allocated()/1e9:.2f}GB")


사진 한 장 크기가 `(518, 294)` 처럼 나옵니다. 원본이 1920×1080 이어도 이렇게 줄어서
들어갑니다. 모델이 정해 둔 표에서 비율에 맞는 크기를 고르기 때문이고, 마음대로 못 올립니다.
**책상 위 케이블처럼 가는 것이 뭉개져 보이는 까닭이 여기 있습니다.**

## 6. 나온 것이 쓸 만한지 봅니다

두 가지를 봅니다.

**첫째, 멀리 튄 점을 버립니다.** 창밖이나 거울에 비친 것은 엉뚱하게 먼 거리로 잡힙니다.
이걸 그냥 두면 장면이 한없이 커지고, 뒤에서 알갱이 크기를 정할 때 기준이 망가집니다.

**둘째, 카메라가 실제로 움직였는지 봅니다.** 카메라가 움직인 폭을 장면 크기로 나눈 값이
0.1 보다 작으면 제자리에서 돌기만 한 것입니다. 그러면 결과가 대개 무너집니다.


In [ ]:
import numpy as np

pts, cols, cams = [], [], []
for v in pred:
    m = v['mask'][0, ..., 0].cpu().numpy()
    pts.append(v['pts3d'][0].cpu().numpy()[m])
    cols.append((v['img_no_norm'][0].cpu().numpy() * 255).clip(0, 255).astype(np.uint8)[m])
    cams.append(v['camera_poses'][0].cpu().numpy())
xyz = np.concatenate(pts); rgb = np.concatenate(cols); cams = np.stack(cams)
print(f"점 {len(xyz):,}개")

lo = np.percentile(xyz, 0.5, axis=0); hi = np.percentile(xyz, 99.5, axis=0)
keep = np.all((xyz > lo) & (xyz < hi), axis=1)
xyz, rgb = xyz[keep], rgb[keep]
print(f"멀리 튄 점을 버리고 {len(xyz):,}개")

cpos = cams[:, :3, 3]
spread = float(np.linalg.norm(cpos.max(0) - cpos.min(0)))
diag = float(np.linalg.norm(xyz.max(0) - xyz.min(0)))
print(f"카메라 퍼짐 {spread:.2f} / 장면 대각선 {diag:.2f} = {spread/diag:.4f}")
print("걸어 다니며 찍은 것으로 보입니다" if spread/diag > 0.1 else
      "카메라가 거의 안 움직였습니다. 걸어 다니며 다시 찍는 편이 낫습니다")

np.save('/content/xyz.npy', xyz); np.save('/content/rgb.npy', rgb)


## 7. 점을 알갱이로 바꿉니다

점은 크기가 없어서 화면을 못 채웁니다. 확대하면 사이가 다 비어 보입니다.
그래서 점 하나를 **부피와 투명도를 가진 알갱이 하나**로 바꿉니다. 이것이 가우시안 스플랫입니다.

알갱이 하나가 담는 값은 원래 **62개**입니다.

| 칸 | 개수 | 어디서 오나 |
|---|---|---|
| 자리 | 3 | 점에서 옵니다 |
| 법선 | 3 | 안 씁니다. 0 으로 둡니다 |
| 기본 색 | 3 | 점 색에서 옵니다 |
| 각도별 색 | 45 | **없습니다.** 학습을 걸어야 채워집니다 |
| 불투명도 | 1 | 우리가 정합니다 |
| 크기 | 3 | 우리가 정합니다 |
| 회전 | 4 | 근거가 없어 안 돌린 값을 넣습니다 |

**각도별 색 45칸을 아예 안 적습니다.** 0 으로 채워 적으면 파일만 3.6배가 됩니다.
그래서 우리 파일은 칸이 17개이고 **알갱이 하나에 68바이트**입니다.

**알갱이 크기를 무엇으로 정하나.** 알갱이 하나가 덮는 넓이는 반지름의 제곱에 비례하니,
점 N 개로 표면을 덮으려면 반지름이 N 의 제곱근에 반비례해야 합니다.

```
r = K x (장면의 가장 긴 변) / sqrt(N)
```

`K` 를 1.4 로 두었습니다. **구멍이 보이면 올리고, 뭉개지면 내리십시오.**


In [ ]:
SH_C0 = 0.28209479177387814
K = 1.4            # 알갱이 크기. 구멍이 보이면 올리고, 뭉개지면 내립니다
OPACITY = 0.9

def to_splat_ply(xyz, rgb, path, K=1.4, opacity=0.9):
    n = len(xyz)
    extent = float((xyz.max(0) - xyz.min(0)).max())
    r = K * extent / np.sqrt(n)

    # 이 모델은 Y 가 아래를 향하고 three.js 는 Y 가 위입니다.
    # y 와 z 의 부호를 뒤집지 않으면 방이 거꾸로 섭니다.
    p = xyz.astype(np.float32).copy()
    p[:, 1] *= -1.0
    p[:, 2] *= -1.0

    out = np.zeros((n, 17), dtype=np.float32)
    out[:, 0:3] = p
    out[:, 6:9] = (rgb.astype(np.float32) / 255.0 - 0.5) / SH_C0   # 색
    out[:, 9] = np.log(opacity / (1.0 - opacity))                  # 불투명도(로짓)
    out[:, 10:13] = np.log(r)                                      # 세 축 반지름(로그)
    out[:, 13] = 1.0                                               # 안 돌린 사원수

    hdr = ("ply\nformat binary_little_endian 1.0\n"
           f"element vertex {n}\n"
           "property float x\nproperty float y\nproperty float z\n"
           "property float nx\nproperty float ny\nproperty float nz\n"
           "property float f_dc_0\nproperty float f_dc_1\nproperty float f_dc_2\n"
           "property float opacity\n"
           "property float scale_0\nproperty float scale_1\nproperty float scale_2\n"
           "property float rot_0\nproperty float rot_1\n"
           "property float rot_2\nproperty float rot_3\n"
           "end_header\n")
    with open(path, 'wb') as f:
        f.write(hdr.encode()); f.write(out.tobytes())
    return r, extent, len(hdr)

t = time.time()
r, extent, hdr_len = to_splat_ply(xyz, rgb, '/content/scene.ply', K=K, opacity=OPACITY)
size = os.path.getsize('/content/scene.ply')
print(f"알갱이 {len(xyz):,}개, 반지름 {r:.5f} (장면 긴 변 {extent:.2f}), {time.time()-t:.1f}초")
print(f"scene.ply {size/1e6:.1f}MB, 알갱이 하나에 {(size-hdr_len)/len(xyz):.0f}바이트")


## 8. 파일을 줄입니다

**이 걸음은 건너뛸 수 없습니다.** 방금 만든 파일은 이백 메가가 넘는데
깃허브는 **파일 하나에 100MB** 가 한계입니다. 올릴 수가 없습니다.

`.sog` 로 바꾸면 십 배 넘게 줄어듭니다. 알갱이 백만 개 기준으로 재 본 값입니다.

| 형식 | 크기 | 배수 |
|---|---|---|
| `.ply` 무압축 | 68.0MB | 1배 |
| `.splat` | 32.0MB | 2.1배 |
| `.compressed.ply` | 16.3MB | 4.2배 |
| **`.sog`** | **5.0MB** | **13.6배** |

값의 자릿수를 줄여 저장하기 때문에 줄어듭니다. 뭉갤수록 작아지고 결과는 조금씩 어긋납니다.


In [ ]:
t = time.time()
!npx -y @playcanvas/splat-transform@latest -w /content/scene.ply /content/scene.sog
print(f"변환 {time.time()-t:.0f}초")

a = os.path.getsize('/content/scene.ply'); b = os.path.getsize('/content/scene.sog')
print(f"{a/1e6:.1f}MB -> {b/1e6:.1f}MB  ({a/b:.1f}배 줄었습니다)")


## 9. 게임에 쓰려면 메시가 하나 더 필요합니다

여기서부터는 **웹에 띄우기만 할 것이면 건너뛰어도 됩니다.**

알갱이는 그리기 위한 것이라 **부딪히지 않습니다.** 캐릭터를 올려 놓고 걸어 다니게
하려면 딛고 설 면이 있어야 하고, 그 면은 삼각형으로 된 메시여야 합니다.

**포아송 표면 복원**을 씁니다. 점에서 닫힌 면을 지어내는 방식이라 안 찍힌 구멍을
알아서 메웁니다. 대신 점이 없는 곳에도 면을 만들어 내므로, 근거가 옅은 곳을 잘라 냅니다.
그다음 삼각형 수를 줄이면 파일이 가벼워집니다.

몇 분 걸립니다. GPU 를 안 쓰고 CPU 로만 도는 걸음입니다.


In [ ]:
import open3d as o3d

xyz = np.load('/content/xyz.npy').astype(np.float64)
rgb = np.load('/content/rgb.npy').astype(np.float64) / 255.0
xyz[:, 1] *= -1.0        # 앞에서와 같은 뒤집기
xyz[:, 2] *= -1.0

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(xyz)
pcd.colors = o3d.utility.Vector3dVector(rgb)

extent = float((xyz.max(0) - xyz.min(0)).max())
voxel = extent / 900.0
pcd = pcd.voxel_down_sample(voxel_size=voxel)
o3d.io.write_point_cloud('/content/room_points.ply', pcd)
print(f"격자로 줄여 점 {len(pcd.points):,}개")

pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel * 4, max_nn=30))
# 방 안에서 찍었으니 법선이 방 가운데를 보게 맞춥니다. 어긋나면 면이 뒤집혀 나옵니다.
pcd.orient_normals_towards_camera_location(pcd.get_center())

t = time.time()
mesh, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=10)
dens = np.asarray(dens)
print(f"포아송 {time.time()-t:.0f}초, 삼각형 {len(mesh.triangles):,}개")

mesh.remove_vertices_by_mask(dens < np.quantile(dens, 0.06))
mesh.remove_unreferenced_vertices()
print(f"근거가 옅은 곳을 자르고 {len(mesh.triangles):,}개")

if len(mesh.triangles) > 180_000:
    mesh = mesh.simplify_quadric_decimation(180_000)
    print(f"단순화 후 {len(mesh.triangles):,}개")

mesh.remove_degenerate_triangles(); mesh.remove_duplicated_vertices()
mesh.remove_non_manifold_edges(); mesh.compute_vertex_normals()
o3d.io.write_triangle_mesh('/content/room_mesh.ply', mesh)
print(f"room_mesh.ply {os.path.getsize('/content/room_mesh.ply')/1e6:.1f}MB")


메시를 웹에서 쓰려면 `.glb` 가 편합니다. 색을 정점에 실어 내보냅니다.


In [ ]:
import trimesh
m = trimesh.Trimesh(
    vertices=np.asarray(mesh.vertices),
    faces=np.asarray(mesh.triangles),
    vertex_colors=(np.asarray(mesh.vertex_colors) * 255).astype(np.uint8),
    process=False)
m.export('/content/room_mesh.glb')
print(f"room_mesh.glb {os.path.getsize('/content/room_mesh.glb')/1e6:.1f}MB")


## 10. 내려받습니다


In [ ]:
files.download('/content/scene.sog')      # 웹에 띄울 알갱이 파일
files.download('/content/room_mesh.glb')  # 게임에 쓸 메시 (안 만들었으면 지우십시오)
print(f"\n전 과정 {time.time()-T0:.0f}초")


## 11. 웹에 올립니다

깃허브에 저장소를 하나 만들고 세 파일을 끌어다 놓으면 끝납니다.

1. 깃허브에서 **New repository** 를 누르고 **Public** 으로 만듭니다
2. `index.html`, `scene.sog`, 그리고 빈 파일 `.nojekyll` 을 끌어다 놓고 커밋합니다
3. **Settings → Pages** 에서 Source 를 `main` 브랜치 루트로 두고 저장합니다
4. 일 분쯤 뒤 `https://<아이디>.github.io/<저장소이름>/` 이 열립니다

`.nojekyll` 은 깃허브가 파일 이름을 임의로 손대지 않게 하는 표시입니다.

띄우는 페이지는 아래 저장소의 `index.html` 을 그대로 쓰셔도 됩니다.
https://github.com/SunCreation/room-3d-demo

**띄울 때 걸리는 함정이 둘 있습니다.**

- **`SparkRenderer` 를 장면에 넣지 않으면** 알갱이가 다 올라가고도 화면이 까맣습니다.
  오류도 안 납니다
- **`.sog` 의 안쪽 이름은 `pcsogszip`** 입니다. 확장자와 달라서 그대로 적으면 안 읽힙니다
